# ML-11 addendum — Does the ranking hold when the label is genuinely in the future?[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanBuilds/Rayanflyrank/blob/main/work/notebooks/w08_forward_window_validation.ipynb?flush_cache=true)**Why this notebook exists.** The capstone paper states one limitation four separate times: the starterCSV has **no date column**, so every number in it is *concurrent* decline detection — "this pageresembles the pages measured as declining" — and never a forecast. A time-aware split is impossiblefrom that file.The warehouse release removes that excuse. `fact_content_daily_performance` is 78,835,655 rows at`report_date x client_hash_id x content_hash_id`, spanning **2025-01-27 to 2026-06-30**, and thelane guide lists it for exactly this: *"time-series features, trend labels, forward-window validation."***The question this notebook answers, and nothing else:** take the shipped hybrid ranking — rule bandselects, logistic regression orders within it — and score it against a label that lives strictly in the*future* of every feature it sees. Does precision@50 survive?## The designA decision point **T**. Everything the model sees comes from on or before T. The label is measuredafter T.| Window | Span | Role ||---|---|---|| Feature window | `T-89 .. T` | 90 days, mirroring the starter's 90-day aggregates || Prior window | `T-29 .. T` | the denominator of the label ratio — known at T, so it is a legal feature || **Label window** | `T+1 .. T+30` | **the future.** Never a feature, never touched during development |`is_declining_fwd = 1` when forward-30-day impressions fall more than 20% below the prior 30 days —the same +/-20% threshold the starter's `trend_direction` uses, shifted forward so it becomes agenuine past -> future label.## Two evaluations, and why the second one is sealedThe data skill is explicit: `fact_content_daily_performance_sample` is the panel's **final month**(June 2026), so developing label logic there means developing inside your own outcome window. Thisnotebook therefore never uses the sample table for label logic, and splits the work in two:| Pass | T | Label window | Purpose ||---|---|---|---|| **DEV** | 2026-02-28 | March 2026 | mid-panel. Every design decision is made here. || **SEALED** | 2026-05-31 | June 2026 | run **once**, at the end, behind an explicit flag. |The capstone paper currently says *"Sealed-evaluation claim: still none."* If the sealed cell below isrun exactly once and its result reported whatever it says, that sentence can finally change — and ifthe result is worse than the paper's 0.900, that is the finding and it gets published as-is.> **Cost discipline.** Every heavy scan caches to `work/outputs/`. Reruns read the cache. The full> table is scanned once per window, not once per cell — repeated 79M-row scans earn HTTP 429.

In [ ]:
# Setup: work from the repo root so this runs on Colab AND from a fresh local clone.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/KhanBuilds/Rayanflyrank"
REPO_DIR = "Rayanflyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

import json, getpass
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import duckdb

RANDOM_STATE = 42                 # same seed as every other notebook in this repo
np.random.seed(RANDOM_STATE)

OUT_DIR = Path("work/outputs")
CACHE_DIR = OUT_DIR / "warehouse_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Token order: env var -> Colab Secret -> prompt. NEVER paste a token into a cell (public repo).
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "need a Hugging Face READ token with gate access"

print("Working dir:", os.getcwd())
print(f"duckdb {duckdb.__version__} | pandas {pd.__version__} | numpy {np.__version__} | sklearn {sklearn.__version__}")

## 1. Connect, and confirm we are pointed at the right release`COUNT(*)` and `MIN/MAX(report_date)` over Parquet read **metadata, not data** — they are near-free andthey are the cheapest possible proof that the path is right. Every number below is asserted against thefigures published in the lane guide, so a wrong path or a re-cut release fails loudly here rather thansilently producing a plausible-looking result 200 lines later.

In [ ]:
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
BUILD_ID = "flyrank_pseudonymized_warehouse_release_v20260703"

T_DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
T_DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
T_FACT_DAILY  = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

# Published counts (lane guide section 2). Asserted, not trusted.
EXPECTED = {"dim_clients": 104, "dim_content": 519_606, "fact_daily": 78_835_655}

counts = {
    "dim_clients": con.sql(f"SELECT COUNT(*) FROM {T_DIM_CLIENTS}").fetchone()[0],
    "dim_content": con.sql(f"SELECT COUNT(*) FROM {T_DIM_CONTENT}").fetchone()[0],
    "fact_daily":  con.sql(f"SELECT COUNT(*) FROM {T_FACT_DAILY}").fetchone()[0],
}
for name, n in counts.items():
    flag = "OK " if n == EXPECTED[name] else "!! "
    print(f"{flag}{name:14} {n:>12,}  (expected {EXPECTED[name]:,})")

date_lo, date_hi = con.sql(f"SELECT MIN(report_date), MAX(report_date) FROM {T_FACT_DAILY}").fetchone()
print(f"\nreport_date spans {date_lo} .. {date_hi}")

assert counts == EXPECTED, "row counts do not match the published release - wrong path or a new build"
assert str(date_lo) == "2025-01-27" and str(date_hi) == "2026-06-30", "date span moved; re-read the guide"
print(f"\nbuild confirmed: {BUILD_ID}")

## 2. Schema discovery — never guess a column nameThe starter CSV and the warehouse do not share column names (`avg_position` there,`gsc_avg_position` here; `client_id` there, `client_hash_id` here). Rather than hard-code guesses thatfail 79 million rows into a scan, this cell **reads the schema** and resolves each field it needs,printing exactly what it found.Two fields are optional: the rule's `stale_90d` and `mature_page` conditions need contenttimestamps from `dim_content`. If the release does not carry them, the rule is rebuilt from theconditions that *are* available and the notebook says so — a reduced rule honestly reported beats asilently different one.

In [ ]:
def columns_of(table_sql):
    return [r[0] for r in con.sql(f"DESCRIBE SELECT * FROM {table_sql} LIMIT 0").fetchall()]

cols_fact = columns_of(T_FACT_DAILY)
cols_content = columns_of(T_DIM_CONTENT)
cols_clients = columns_of(T_DIM_CLIENTS)

print("fact_content_daily_performance columns:")
print("  " + ", ".join(cols_fact))
print("\ndim_content columns:")
print("  " + ", ".join(cols_content))
print("\ndim_clients columns:")
print("  " + ", ".join(cols_clients))


def pick(candidates, available, what, required=True):
    """Resolve the first candidate that exists; report what was chosen."""
    for c in candidates:
        if c in available:
            return c
    if required:
        raise KeyError(f"none of {candidates} found for {what}; available: {available}")
    print(f"  (optional) {what}: NOT AVAILABLE - tried {candidates}")
    return None

print("\nresolved:")
C = {}
C["date"]        = pick(["report_date"], cols_fact, "report_date")
C["client"]      = pick(["client_hash_id"], cols_fact, "client key")
C["content"]     = pick(["content_hash_id"], cols_fact, "content key")
C["impressions"] = pick(["gsc_impressions", "impressions"], cols_fact, "impressions")
C["clicks"]      = pick(["gsc_clicks", "clicks"], cols_fact, "clicks")
C["position"]    = pick(["gsc_avg_position", "avg_position"], cols_fact, "position")
C["created"]     = pick(["content_created_at", "created_at", "first_published_at"], cols_content,
                        "content creation date", required=False)
C["updated"]     = pick(["content_updated_at", "updated_at", "last_modified_at"], cols_content,
                        "content update date", required=False)
C["gsc_start"]   = pick(["gsc_data_start"], cols_clients, "gsc history start")

for k, v in C.items():
    if v:
        print(f"  {k:12} -> {v}")

HAS_AGE = C["created"] is not None
HAS_STALE = C["updated"] is not None
RULE_CONDITIONS = ["established_coverage", "has_demand", "mid_position"] \
                  + (["stale_90d"] if HAS_STALE else []) + (["mature_page"] if HAS_AGE else [])
MAX_SCORE = 3 + 2 + 2 + (1 if HAS_STALE else 0) + (1 if HAS_AGE else 0)
print(f"\nrule conditions reconstructable: {RULE_CONDITIONS}  (max score {MAX_SCORE}, starter max 9)")

## 3. The windows, and who is eligible to be in themTwo guards, both of which would otherwise corrupt the label:1. **Client history depth.** The panel is unbalanced — history depth differs wildly per client. A client   whose tracking began inside the feature window would show "no traffic" that is really "no tracking   yet". Only clients whose `gsc_data_start` predates the feature window are eligible.2. **Client survival.** If a client stops reporting entirely after T, every one of its pages looks like a   total collapse. That is client churn, not page decay. Clients must still be reporting in the label   window to be scored.Both are measured and printed below, not assumed.

In [ ]:
from datetime import date, timedelta

FEATURE_DAYS = 90       # mirrors the starter's 90-day aggregates
LABEL_DAYS = 30         # mirrors the starter's 30-day trend window
DECLINE_THRESHOLD = 0.8 # -20%, the same threshold the starter's trend_direction uses
MIN_PRIOR_IMPRESSIONS = 10   # a ratio needs a denominator; below this the label is noise

T_DEV = date(2026, 2, 28)    # mid-panel. All development happens here.
T_SEALED = date(2026, 5, 31) # the final month is the outcome window. Run once, at the end.

for tag, T in [("DEV", T_DEV), ("SEALED", T_SEALED)]:
    print(f"{tag:7} T={T}  features {T - timedelta(days=FEATURE_DAYS - 1)} .. {T}"
          f"   label {T + timedelta(days=1)} .. {T + timedelta(days=LABEL_DAYS)}")

# Eligibility, measured per pass.
elig = con.sql(f"""
    SELECT {C['client']} AS client_hash_id, {C['gsc_start']} AS gsc_start
    FROM {T_DIM_CLIENTS}
    WHERE {C['gsc_start']} IS NOT NULL
""").df()
print(f"\nclients with a known GSC start: {len(elig)} of {counts['dim_clients']}")
for tag, T in [("DEV", T_DEV), ("SEALED", T_SEALED)]:
    cutoff = pd.Timestamp(T - timedelta(days=FEATURE_DAYS))
    n = (pd.to_datetime(elig["gsc_start"]) <= cutoff).sum()
    print(f"  {tag:7} clients with history predating the feature window: {n}")

## 4. One SQL builder, used for both windowsThe whole point of the warehouse workflow: **send the question to the rows.** This aggregates78.8M daily rows down to one row per content item entirely inside DuckDB, and brings back only thesmall result. Nothing here loads a multi-million-row frame into pandas.Three things the SQL is careful about:- **The feature/label boundary is enforced in SQL**, not by convention. Every feature aggregate is  wrapped in `CASE WHEN report_date <= T`, and the label aggregate in `CASE WHEN report_date > T`.  A cell below re-asserts the separation independently.- **A concurrent label is computed alongside the forward one**, from the two 30-day windows *before* T.  That is the starter's definition, rebuilt here — which makes the comparison in section 6 an  apples-to-apples measurement of what "forecasting instead of detecting" actually costs.- **`avg_position` averages only over days with impressions.** A day with no impressions carries no  position reading, and averaging in its zero would recreate exactly the `avg_position = 0` trap the  data contract already caught in the starter file.

In [ ]:
def build_frame(T, tag, force=False, include_label=True):
    """Aggregate the daily fact into one row per content item, for decision point T.

    include_label=False truncates the scan at T entirely - no post-T row is even read. That is
    what makes the boundary check below a real test rather than a restatement.
    Cached to parquet: the 79M-row scan happens once per window, not once per rerun.
    """
    cache = CACHE_DIR / f"forward_window_{tag.lower()}_{T.isoformat()}.parquet"
    if cache.exists() and not force:
        out = pd.read_parquet(cache)
        print(f"[cache] {cache}  ({len(out):,} rows)")
        return out

    feat_start = T - timedelta(days=FEATURE_DAYS - 1)
    prior_start = T - timedelta(days=LABEL_DAYS - 1)          # T-29 .. T
    prev_start = T - timedelta(days=2 * LABEL_DAYS - 1)       # T-59 .. T-30 (concurrent comparison)
    prev_end = T - timedelta(days=LABEL_DAYS)
    label_end = T + timedelta(days=LABEL_DAYS) if include_label else T
    scan_end = label_end

    d, cl, ct = C["date"], C["client"], C["content"]
    imp, clk, pos = C["impressions"], C["clicks"], C["position"]

    sql = f"""
    WITH eligible_clients AS (
        SELECT {cl} FROM {T_DIM_CLIENTS}
        WHERE {C['gsc_start']} IS NOT NULL
          AND {C['gsc_start']} <= DATE '{feat_start}'
    ),
    surviving_clients AS (
        SELECT DISTINCT {cl} FROM {T_FACT_DAILY}
        WHERE {d} > DATE '{T}' AND {d} <= DATE '{T + timedelta(days=LABEL_DAYS)}'
    ),
    agg AS (
        SELECT
            f.{cl} AS client_hash_id,
            f.{ct} AS content_hash_id,
            -- ---------- FEATURE WINDOW: strictly on or before T ----------
            SUM(CASE WHEN f.{d} >= DATE '{feat_start}' AND f.{d} <= DATE '{T}'
                     THEN f.{imp} ELSE 0 END)                                  AS impressions_90d,
            SUM(CASE WHEN f.{d} >= DATE '{feat_start}' AND f.{d} <= DATE '{T}'
                     THEN f.{clk} ELSE 0 END)                                  AS clicks_90d,
            COUNT(DISTINCT CASE WHEN f.{d} >= DATE '{feat_start}' AND f.{d} <= DATE '{T}'
                                AND f.{imp} > 0 THEN f.{d} END)                AS days_with_impressions,
            AVG(CASE WHEN f.{d} >= DATE '{feat_start}' AND f.{d} <= DATE '{T}'
                     AND f.{imp} > 0 THEN f.{pos} END)                         AS avg_position,
            STDDEV_SAMP(CASE WHEN f.{d} >= DATE '{feat_start}' AND f.{d} <= DATE '{T}'
                             AND f.{imp} > 0 THEN f.{pos} END)                 AS position_sd,
            SUM(CASE WHEN f.{d} >= DATE '{prior_start}' AND f.{d} <= DATE '{T}'
                     THEN f.{imp} ELSE 0 END)                                  AS imp_prior30,
            SUM(CASE WHEN f.{d} >= DATE '{prev_start}' AND f.{d} <= DATE '{prev_end}'
                     THEN f.{imp} ELSE 0 END)                                  AS imp_prev30,
            -- ---------- LABEL WINDOW: strictly after T ----------
            SUM(CASE WHEN f.{d} > DATE '{T}' AND f.{d} <= DATE '{label_end}'
                     THEN f.{imp} ELSE 0 END)                                  AS imp_fwd30
        FROM {T_FACT_DAILY} f
        JOIN eligible_clients e  ON e.{cl} = f.{cl}
        JOIN surviving_clients s ON s.{cl} = f.{cl}
        WHERE f.{d} >= DATE '{prev_start}' AND f.{d} <= DATE '{scan_end}'
        GROUP BY 1, 2
    )
    SELECT * FROM agg
    WHERE imp_prior30 >= {MIN_PRIOR_IMPRESSIONS}
    """
    print(f"scanning the daily fact for T={T} ... (one pass, then cached)")
    out = con.sql(sql).df()

    if HAS_AGE or HAS_STALE:
        sel = [C["content"]] + [c for c in (C["created"], C["updated"]) if c]
        meta = con.sql(f"SELECT {', '.join(sel)} FROM {T_DIM_CONTENT}").df()
        meta = meta.rename(columns={C["content"]: "content_hash_id"})
        out = out.merge(meta, on="content_hash_id", how="left")
        if HAS_AGE:
            out["content_age_days"] = (pd.Timestamp(T) - pd.to_datetime(out[C["created"]])).dt.days
        if HAS_STALE:
            out["days_since_last_update"] = (pd.Timestamp(T) - pd.to_datetime(out[C["updated"]])).dt.days

    out.to_parquet(cache, index=False)
    print(f"[wrote] {cache}  ({len(out):,} rows)")
    return out

In [ ]:
dev = build_frame(T_DEV, "dev")

# --- grain probe: one row per content item, or every aggregate below is wrong ---
dupes = dev.groupby("content_hash_id").size()
assert (dupes == 1).all(), f"grain broken: {(dupes > 1).sum()} content items appear more than once"

# --- the two labels ---
dev["is_declining_fwd"] = (dev["imp_fwd30"] < DECLINE_THRESHOLD * dev["imp_prior30"]).astype(int)
dev["is_declining_conc"] = (dev["imp_prior30"] < DECLINE_THRESHOLD * dev["imp_prev30"]).astype(int)

print(f"content items:            {len(dev):,}")
print(f"clients:                  {dev['client_hash_id'].nunique()}")
print(f"forward   base rate:      {dev['is_declining_fwd'].mean():.4f}")
print(f"concurrent base rate:     {dev['is_declining_conc'].mean():.4f}   (starter file: 0.5421)")
print(f"the two labels agree on:  {(dev['is_declining_fwd'] == dev['is_declining_conc']).mean():.4f} of rows")
print("\n-> if that agreement were ~1.0 the forward label would be a relabelling, not a new question.")

### The check that matters most: no feature may see the futureThe whole claim of this notebook rests on the feature/label boundary holding. Convention is notevidence, so it is tested directly: rebuild the frame with the label window *truncated away* andconfirm every feature column is bit-for-bit identical. If a single aggregate leaked across T, thisfails.

In [ ]:
FEATURE_COLS_RAW = ["impressions_90d", "clicks_90d", "days_with_impressions",
                    "avg_position", "position_sd", "imp_prior30", "imp_prev30"]

# Rebuild with the label window removed entirely: the scan itself stops at T, so no post-T row is
# even read. If any feature aggregate silently reaches past T, its value here will differ.
probe = build_frame(T_DEV, "leakcheck", include_label=False)

merged = dev[["content_hash_id"] + FEATURE_COLS_RAW].merge(
    probe[["content_hash_id"] + FEATURE_COLS_RAW], on="content_hash_id", suffixes=("", "_probe"))
mismatch = {c: int((~np.isclose(merged[c].fillna(-1), merged[f"{c}_probe"].fillna(-1))).sum())
            for c in FEATURE_COLS_RAW}
print("feature columns that change when the label window is removed:")
for c, n in mismatch.items():
    print(f"  {'OK  ' if n == 0 else 'LEAK'} {c:24} {n:,} rows differ")
assert all(n == 0 for n in mismatch.values()), "a feature aggregate reaches past T - fix the SQL"
print("\nfeature/label boundary holds: no feature column depends on post-T data.")

## 5. Rebuild the shipped hybrid on warehouse featuresSame construction as ML-08, ported to the warehouse's columns: the five-condition rule selects theband and supplies the reason codes, a logistic regression orders pages inside it, and the two combinethrough the same key — `band * 1000 + probability * 100`, so the band dominates and the model onlybreaks ties within it.Seed 42, `GroupKFold(5)` on `client_hash_id`, counts `log1p`'d, `predict_proba` used as a rankingscore and never thresholded. Nothing here is re-tuned for the warehouse: re-tuning would make thecomparison against the paper's 0.900 meaningless.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.base import clone

CAPACITY_PER_SPRINT = 50
K_VALUES = [10, 20, 50, 100, 200, 500]


def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())


def roc_auc(labels, scores):
    labels = np.asarray(labels)
    n_pos, n_neg = labels.sum(), (1 - labels).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = pd.Series(np.asarray(scores, dtype=float)).rank().to_numpy()
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def build_rule(frame):
    """The ML-07 rule, rebuilt from whichever conditions the warehouse schema supports."""
    established = frame["days_with_impressions"].between(20, 87)
    demand = frame["impressions_90d"] >= 40
    midpos = frame["avg_position"].gt(3) & frame["avg_position"].le(50)
    score = 3 * established.astype(int) + 2 * demand.astype(int) + 2 * midpos.astype(int)
    codes = (np.where(established, "established_coverage;", "")
             + np.where(demand, "has_demand;", "")
             + np.where(midpos, "mid_position;", ""))
    if HAS_STALE:
        stale = frame["days_since_last_update"] >= 90
        score = score + stale.fillna(False).astype(int)
        codes = codes + np.where(stale.fillna(False), "stale_90d;", "")
    if HAS_AGE:
        mature = frame["content_age_days"].between(90, 364)
        score = score + mature.fillna(False).astype(int)
        codes = codes + np.where(mature.fillna(False), "mature_page;", "")
    return score.to_numpy(), np.where(codes == "", "no_signal", codes)


ALL_FEATURES = ["impressions_90d", "clicks_90d", "days_with_impressions",
                "avg_position", "position_sd", "imp_prior30", "imp_prev30"]

# The concurrent label IS a threshold on imp_prior30 / imp_prev30. Leaving that pair in the matrix
# reconstructs it exactly - the identical trap ML-05 caught in the starter file (1.0000 agreement).
# So the pair is excluded whenever the concurrent label is the target.
RECONSTRUCTS_CONCURRENT = ("imp_prior30", "imp_prev30")


def build_matrix(frame, exclude=()):
    keep = [c for c in ALL_FEATURES if c not in exclude]
    X = frame[keep].copy()
    X["has_position"] = X["avg_position"].notna().astype(int)
    X["has_position_sd"] = X["position_sd"].notna().astype(int)
    for c in ["impressions_90d", "clicks_90d", "imp_prior30", "imp_prev30"]:
        if c in X.columns:
            X[c] = np.log1p(X[c])
    return X


def make_pipe():
    """Every column here is numeric, so no ColumnTransformer is needed - impute, scale, fit."""
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ])


def hybrid_key(band, model_score):
    """Rule band dominates; the model only orders inside it. Identical to ML-08."""
    return band * 1000 + model_score * 100


def evaluate(frame, label_col, tag, exclude=()):
    y = frame[label_col].to_numpy()
    groups = frame["client_hash_id"].to_numpy()
    X = build_matrix(frame, exclude=exclude)
    band, codes = build_rule(frame)
    base = float(y.mean())

    oof = np.full(len(frame), np.nan)
    per_fold = {"rule": [], "logistic": [], "hybrid": []}
    for tr, te in GroupKFold(n_splits=5).split(X, y, groups):
        pipe = clone(make_pipe()).fit(X.iloc[tr], y[tr])
        p = pipe.predict_proba(X.iloc[te])[:, 1]
        oof[te] = p
        per_fold["rule"].append(precision_at_k(y[te], band[te], CAPACITY_PER_SPRINT))
        per_fold["logistic"].append(precision_at_k(y[te], p, CAPACITY_PER_SPRINT))
        per_fold["hybrid"].append(
            precision_at_k(y[te], hybrid_key(band[te], p), CAPACITY_PER_SPRINT))
    assert not np.isnan(oof).any(), "a row was never scored out-of-fold"

    hyb = hybrid_key(band, oof)
    systems = {
        "flag_everything_floor": {f"p@{k}": base for k in K_VALUES} | {"roc_auc": 0.5},
        "rule_baseline":  {f"p@{k}": precision_at_k(y, band, k) for k in K_VALUES} | {"roc_auc": roc_auc(y, band)},
        "logistic":       {f"p@{k}": precision_at_k(y, oof, k) for k in K_VALUES} | {"roc_auc": roc_auc(y, oof)},
        "hybrid_shipped": {f"p@{k}": precision_at_k(y, hyb, k) for k in K_VALUES} | {"roc_auc": roc_auc(y, hyb)},
    }
    print(f"\n=== {tag} | label={label_col} | n={len(frame):,} | "
          f"clients={frame['client_hash_id'].nunique()} | base rate={base:.4f} ===")
    print(f"features ({X.shape[1]}): {list(X.columns)}")
    if exclude:
        print(f"excluded as label-reconstructing: {list(exclude)}")
    header = f"{'system':<24}" + "".join(f"{'p@'+str(k):>9}" for k in K_VALUES) + f"{'AUC':>9}"
    print(header)
    for name, m in systems.items():
        print(f"{name:<24}" + "".join(f"{m['p@'+str(k)]:>9.3f}" for k in K_VALUES) + f"{m['roc_auc']:>9.4f}")
    return {"base_rate": base, "n": int(len(frame)),
            "clients": int(frame["client_hash_id"].nunique()),
            "features": list(X.columns), "excluded": list(exclude),
            "systems": systems, "per_fold_precision_at_50": per_fold,
            "scores": {"band": band, "oof": oof, "hybrid": hyb, "y": y}}


# The headline forward model. imp_prior30 is the label's DENOMINATOR and is fully known at T,
# so it is a legal feature here - only the forward window is withheld.
dev_fwd = evaluate(dev, "is_declining_fwd", "DEV forward-window (headline)")

## 6. The comparison that mattersSame rows, same model, same split. The only thing that changes is **whether the outcome lives in thefuture**.One subtlety decides whether this comparison means anything. The concurrent label is`imp_prior30 < 0.8 x imp_prev30` — so if that pair stays in the feature matrix, the model reconstructsthe label exactly and scores a meaningless ~1.0. That is precisely the trap ML-05 caught in the starterfile, where `impressions_last_30d` / `impressions_prev_30d` reproduced `trend_direction` with 1.0000agreement. So the pair is dropped for the concurrent target.That leaves two feature sets, so the notebook reports **three rows** rather than pretending onecomparison covers it:| Row | Label | Features | Reads as ||---|---|---|---|| A | concurrent | pair dropped | the paper's experiment, rebuilt on warehouse data || B | forward | pair dropped | strictly like-for-like against A || C | forward | `imp_prior30` kept | the honest best forward model — the denominator is known at T |**A vs B is the clean measurement of what forecasting costs.** C is the stronger model, with a caveatthat belongs beside it rather than under it.> **Why C is not simply the better answer: mean reversion.** The forward label is> `imp_fwd30 < 0.8 x imp_prior30`, so `imp_prior30` sits in its *denominator*. A page whose prior> window happened to catch a noise spike is mechanically more likely to "decline" next month, whatever> its underlying health. `imp_prior30` is legitimately known at T — this is not leakage — but a model> holding it can earn precision by detecting reversion to the mean rather than genuine decay. A> synthetic dry run of this notebook's own logic (`work/scripts/dryrun_w08_logic.py`) reproduces the> effect on pure noise, which is how it was found. **Report B as the honest forecasting number and C> as the operational one, and say which is which** — the same discipline Section 5 of the paper> applies to the in-sample rule.

In [ ]:
dev_conc = evaluate(dev, "is_declining_conc", "A. DEV concurrent (paper's definition, rebuilt)",
                    exclude=RECONSTRUCTS_CONCURRENT)
dev_fwd_matched = evaluate(dev, "is_declining_fwd", "B. DEV forward, matched feature set",
                           exclude=RECONSTRUCTS_CONCURRENT)

print("\n\n=== THE COST OF FORECASTING INSTEAD OF DETECTING (hybrid, p@50) ===")
a = dev_conc["systems"]["hybrid_shipped"]["p@50"]
b = dev_fwd_matched["systems"]["hybrid_shipped"]["p@50"]
c = dev_fwd["systems"]["hybrid_shipped"]["p@50"]
print(f"  A concurrent, matched features:  {a:.3f}  vs base {dev_conc['base_rate']:.3f}"
      f"   lift {a / dev_conc['base_rate']:.2f}x")
print(f"  B forward,    matched features:  {b:.3f}  vs base {dev_fwd_matched['base_rate']:.3f}"
      f"   lift {b / dev_fwd_matched['base_rate']:.2f}x")
print(f"  C forward,    imp_prior30 kept:  {c:.3f}  vs base {dev_fwd['base_rate']:.3f}"
      f"   lift {c / dev_fwd['base_rate']:.2f}x")
print(f"\n  cost of forecasting (B - A):     {b - a:+.3f} precision@50")
print(f"  value of the denominator (C - B):{c - b:+.3f} precision@50")
print(f"\n  for reference, the paper's starter-file number was 0.900 vs a 0.5421 base rate (1.66x).")
print("  That was concurrent detection on 30,000 rows and is NOT the same experiment - compare lifts,")
print("  not raw precisions, and only ever alongside each row's own base rate.")


def bootstrap_ci(labels, scores, k=50, n_boot=2000, seed=RANDOM_STATE):
    """95% interval on precision@k, resampling the SELECTED k rows - p@50 is a claim about 50 rows."""
    rng = np.random.default_rng(seed)
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")[:k]
    sel = np.asarray(labels)[order]
    draws = rng.choice(sel, size=(n_boot, k), replace=True).mean(axis=1)
    return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))

lo, hi = bootstrap_ci(dev_fwd["scores"]["y"], dev_fwd["scores"]["hybrid"])
print(f"\n  forward hybrid p@50 = {c:.3f}, 95% bootstrap interval [{lo:.3f}, {hi:.3f}]")
print(f"  clears the base rate? {'YES' if lo > dev_fwd['base_rate'] else 'NO - the interval covers chance'}")

print("\nper-fold p@50 (folds are groups of clients):")
for name, vals in dev_fwd["per_fold_precision_at_50"].items():
    print(f"  {name:<10} {[round(v, 2) for v in vals]}  mean {np.mean(vals):.3f}  worst {min(vals):.3f}")

## 7. The sealed evaluation — run this cell exactly onceEverything above was developed on the March window. This cell scores the **June 2026** window, whichhas not been looked at, and it is the one number in this project that can honestly be called a sealedtest.The rules, stated before running it:- **Run it once.** Not once per tweak. If a bug forces a rerun, say so in the write-up — a sealed test  you re-ran twice is a validation set wearing a disguise.- **Report whatever it says.** If the forward hybrid lands below the base rate, that is the finding and  it goes in the paper next to the 0.900, not instead of it.- **Change nothing after seeing it.** No threshold, no feature, no model choice.Set `RUN_SEALED = True` when the development pass above is final and you are ready to spend it.

In [ ]:
RUN_SEALED = False    # <- flip to True exactly once, when everything above is final

SEALED_RECEIPT = OUT_DIR / "forward_window_sealed.json"

if not RUN_SEALED:
    print("SEALED evaluation not run (RUN_SEALED is False).")
    print("Finish the development pass, then flip the flag - once.")
elif SEALED_RECEIPT.exists():
    print(f"SEALED evaluation ALREADY RUN - receipt exists at {SEALED_RECEIPT}")
    print("Re-running would turn a sealed test into a validation set. Delete the file deliberately,")
    print("and disclose the rerun in the write-up, if you truly must.")
    print(json.dumps(json.load(open(SEALED_RECEIPT))["systems"]["hybrid_shipped"], indent=2))
else:
    sealed = build_frame(T_SEALED, "sealed")
    dupes = sealed.groupby("content_hash_id").size()
    assert (dupes == 1).all(), "grain broken in the sealed frame"
    sealed["is_declining_fwd"] = (
        sealed["imp_fwd30"] < DECLINE_THRESHOLD * sealed["imp_prior30"]).astype(int)

    res = evaluate(sealed, "is_declining_fwd", "SEALED June 2026 (evaluated once)")
    lo_s, hi_s = bootstrap_ci(res["scores"]["y"], res["scores"]["hybrid"])
    p50 = res["systems"]["hybrid_shipped"]["p@50"]
    print(f"\nSEALED hybrid p@50 = {p50:.3f}  95% CI [{lo_s:.3f}, {hi_s:.3f}]  "
          f"base rate {res['base_rate']:.4f}")

    receipt = {k: v for k, v in res.items() if k != "scores"}
    receipt |= {"decision_point": T_SEALED.isoformat(), "bootstrap_95": [lo_s, hi_s],
                "evaluated": "once, sealed - June 2026 was untouched during development",
                "random_state": RANDOM_STATE, "build_id": BUILD_ID}
    SEALED_RECEIPT.write_text(json.dumps(receipt, indent=2, default=float))
    print(f"\nwrote {SEALED_RECEIPT}")

## 8. The receiptSame discipline as the rest of the repo: the numbers this notebook produces are committed as JSON sothe paper can trace every figure back to a file, and so a later rerun fails loudly if it disagrees.The cached parquet frames are **not** committed — they are derived data, and datasets never enter git.

In [ ]:
receipt = {
    "question": "does the shipped hybrid ranking survive a label that lives strictly in the future?",
    "build_id": BUILD_ID,
    "random_state": RANDOM_STATE,
    "design": {
        "decision_point_dev": T_DEV.isoformat(),
        "decision_point_sealed": T_SEALED.isoformat(),
        "feature_window_days": FEATURE_DAYS,
        "label_window_days": LABEL_DAYS,
        "decline_threshold": DECLINE_THRESHOLD,
        "min_prior_impressions": MIN_PRIOR_IMPRESSIONS,
        "split": "GroupKFold(5) on client_hash_id",
        "rule_conditions_available": RULE_CONDITIONS,
        "max_rule_score": MAX_SCORE,
    },
    "guards": {
        "client_history": "clients whose gsc_data_start predates the feature window only",
        "client_survival": "clients must still report in the label window (churn != page decay)",
        "grain": "one row per content_hash_id, asserted",
        "feature_label_boundary": "asserted: no feature column changes when the label window is removed",
    },
    "dev_forward_headline": {k: v for k, v in dev_fwd.items() if k != "scores"},
    "dev_forward_matched_features": {k: v for k, v in dev_fwd_matched.items() if k != "scores"},
    "dev_concurrent": {k: v for k, v in dev_conc.items() if k != "scores"},
    "cost_of_forecasting_p50": float(
        dev_fwd_matched["systems"]["hybrid_shipped"]["p@50"]
        - dev_conc["systems"]["hybrid_shipped"]["p@50"]),
    "starter_comparison": {
        "starter_hybrid_p50_concurrent": 0.900,
        "starter_base_rate": 0.5421,
        "note": "the starter numbers are concurrent detection on 30,000 rows; these are not the same experiment",
    },
    "versions": {"duckdb": duckdb.__version__, "pandas": pd.__version__,
                 "numpy": np.__version__, "scikit-learn": sklearn.__version__},
}
path = OUT_DIR / "forward_window_metrics.json"
path.write_text(json.dumps(receipt, indent=2, default=float))
print(f"wrote {path}")
print(json.dumps(receipt["dev_forward_headline"]["systems"]["hybrid_shipped"], indent=2, default=float))

## 9. What this changes in the paperFill this in **after** running the cells above — from what they actually printed, not from what thisnotebook hoped they would print.- **Forward-window result (DEV, March 2026), row B — matched features:** hybrid p@50 = ____ against a  base rate of ____ (lift ____x). This is the honest forecasting number.- **Row C — with `imp_prior30`:** ____ . Report it as the operational number, with the mean-reversion  caveat from section 6 attached.- **The cost of forecasting (B − A):** ____ points, measured on the same rows and the same features.- **Sealed result (June 2026):** ____, evaluated once.Then, in `docs/index.html`:1. **Section 5** currently ends *"Sealed-evaluation claim: still none."* If the sealed cell ran once,   that sentence changes — and it should state the number whichever way it went.2. **Section 6** currently says the work is *"not a forecast"* and *"a time-aware split remains   impossible from this file."* Both were true of the starter file and are now answered by a second   dataset. Rewrite them to say what was measured, and keep the starter's limitation where it belongs:   as a fact about the starter file.3. **Section 2** should name the second source: the warehouse release, its build id, and the windows.4. The **abstract** stays a 5-sentence summary. If the forward result holds, it belongs in sentence 4.**What this notebook still does not buy.** It does not make anything causal — there is still no recordof what an editor did, so the decision log Section 7 asks for remains the blocker for any claim thatrefreshing a page *causes* recovery. Do not let a bigger dataset launder that distinction.